# Mini-ColBERTv2 on SciFact (Colab GPU)

Run the full pipeline: encode → k-means → 2-bit residual → IVF → search → CAD-MaxSim → evaluate.

In [ ]:
# 1. Clone repo
!git clone https://github.com/HoangNguyen689/colbert-hoangnd.git
%cd colbert-hoangnd

In [ ]:
# 2. Install dependencies (use CUDA wheels for Colab T4)
!pip install -q torch transformers scikit-learn datasets beir tqdm numpy colbert-ai
# For official ColBERT baseline (optional, may need faiss-gpu)
# !pip install -q faiss-git

In [ ]:
# 3. Encode SciFact corpus and queries (uses official colbert-ir/colbertv2.0 checkpoint)
!python -m mini_colbert.encode --data_root data --output_dir outputs --doc_batch_size 64 --query_batch_size 64

In [ ]:
# 4. Build k-means centroids (K=8192)
!python -m mini_colbert.build_index --emb_path outputs/doc_embeddings.pt --output_dir outputs --K 8192 --n_samples 200000

In [ ]:
# 5. 2-bit residual quantization
!python -m mini_colbert.compress_residuals --emb_path outputs/doc_embeddings.pt --centroids_path outputs/centroids.pt --codes_path outputs/codes.pt --output_dir outputs --nbits 2

In [ ]:
# 6. Build IVF
!python -m mini_colbert.ivf --emb_path outputs/doc_embeddings.pt --codes_path outputs/codes.pt --output_dir outputs

In [ ]:
# 7. Approximate search + MaxSim
!python -m mini_colbert.search --index_dir outputs --query_emb outputs/query_embeddings.pt --output outputs/search_results.pt --ncells 4 --topk 100 --device cuda

In [ ]:
# 8. CAD-MaxSim 4 ablations
!python -m mini_colbert.cad_maxsim --index_dir outputs --query_emb outputs/query_embeddings.pt --qrels outputs/qrels_test.json --output_dir outputs --ncells 4 --topk 100 --device cuda

In [ ]:
# 9. Evaluate all variants
import os
for name in ['search_results.pt', 'rankings_original.json', 'rankings_idf.json', 'rankings_adaptive.json', 'rankings_cad.json']:
    print(f'=== {name} ===')
    path = f'outputs/{name}'
    if os.path.exists(path):
        !python -m mini_colbert.evaluate --rankings {path} --qrels outputs/qrels_test.json

In [ ]:
# 10. Compare with official baseline
!python -m mini_colbert.compare_official --rankings outputs/rankings_original.json --qrels outputs/qrels_test.json